# tutorial-02: 社員名簿クレンジング

**pandas のクレンジングだけを、15分で一周する教材です。**

人事システムからエクスポートした社員名簿が1枚あります。
途中で誰かが Excel で触った跡が残っていて、このままでは部門別の人数も人件費も出せません。
**使える状態にするまで**を、ノートブック1枚でやります。

## この教材のゴール

```
社員コード  氏名          部署      入社日        基本給        ← 全部ただの文字列
    0001    山田　太郎    営業部    2019/04/01    320000
       2    佐藤 花子     営業      2020年4月1日  298,000
    0003    鈴木　一郎    ｾｰﾙｽ      2018/10/01    ￥355,000
                              ↓
社員コード  氏名        dept_cd  division    入社日       基本給
    0001    山田 太郎   D01      フロント    2019-04-01      320000  ← 型が付いている
    0002    佐藤 花子   D01      フロント    2020-04-01      298000
    0003    鈴木 一郎   D01      フロント    2018-10-01      355000
```

## 使う名前

同じ語を途中で違う意味に使わないよう、4つだけ名前を決めておきます。

| 名前 | 中身 | 行数 |
| --- | --- | --- |
| `raw` | CSV を読んだだけ。全部文字列 | 20 |
| `dedup` | 重複を落としたもの | 18 |
| `clean` | 型を整え、使える行だけにしたもの | 16 |
| `rejected` | 使えないので隔離した行。**捨てていない** | 2 |

## 進め方

節ごとに「考え方 → セルを実行 → どこを見るか」で進みます。全6節、練習は3問だけです。

**練習の答えは、すぐ下の `<details>` に付いています。** 開いて写してかまいません。
詰まったまま止まるより先へ進んでください。

> この教材が扱うのは**クレンジングまで**です。
> Parquet への書き出し、二度流しても壊れない出力、増分やバックフィルは
> tutorial-01 (POS売上パイプライン) のほうで扱います。

In [ ]:
import unicodedata

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

---

## 1. まず現物を見る

**直し始める前に、何がどう汚れているかを数えます。**
見ずに直すと、直したつもりの列が実は別の壊れ方をしていた、ということが起こります。

読むときのポイントは2つです。

- `dtype=str` … 型を推測させません。推測させると `0001` が `1` になります
- `keep_default_na=False` … 空欄を勝手に `NaN` にさせません。**何が欠損かは自分で決めます**

> 出力の見どころ: `部署` に同じ部署の別表記がいくつ並んでいるか、
> `入社日` に何種類の書式が混ざっているか。

In [ ]:
raw = pd.read_csv("/data/roster_2024-04.csv", dtype=str, keep_default_na=False)

print("行数:", len(raw))
print(raw.head(5))

print("\n部署の書かれ方:")
print(raw["部署"].value_counts())

print("\n入社日の書かれ方 (先頭5件):")
print(raw["入社日"].value_counts().head(5))

---

## 2. 表記を揃えて、欠損を1つにする

最初にやるのは**文字そのものを揃える**ことです。ここを飛ばすと、
後の `groupby` で `セールス` と `ｾｰﾙｽ` が別のものとして数えられます。

`unicodedata.normalize("NFKC", s)` は、全角の英数字・カタカナ・記号を半角に寄せます。
**全角スペースも半角スペースになる**ので、そのあと `strip()` すると前後の空白が落ちます。

次に**欠損の書かれ方を1つにします**。「空欄」「`-`」「`N/A`」「`不明`」は、
どれも「値が無い」という意味で書かれています。これを全部 `pd.NA` に寄せます。
`pd.NA` にしておくと `isna()` で数えられ、集計のときに自動で除かれます。

> 出力の見どころ: `before` と `after` で全角が消えていること。
> `欠損の数` に、社員コード・入社日・基本給・週勤務時間 が1件ずつ出ていること。

In [ ]:
# 「値が無い」の書かれ方。ここに挙げたものだけを欠損として扱う、と決める
NA_TOKENS = ["", "-", "N/A", "不明"]


def norm(s):
    """全角を半角に寄せ、前後の空白を落とす"""
    return unicodedata.normalize("NFKC", s).strip()


print("before:", repr(raw.loc[0, "氏名"]), repr(raw.loc[2, "部署"]), repr(raw.loc[2, "週勤務時間"]))

for col in raw.columns:
    raw[col] = raw[col].map(norm)

raw = raw.replace(NA_TOKENS, pd.NA)

print("after :", repr(raw.loc[0, "氏名"]), repr(raw.loc[2, "部署"]), repr(raw.loc[2, "週勤務時間"]))

print("\n欠損の数:")
print(raw.isna().sum())

In [ ]:
# ✍ 書いてみる: メール列を小文字に揃えたものを ans に入れてください。
#              (大文字が混ざったままだと、メールアドレスを鍵にした突き合わせが外れます)

ans = ...   # ここに書く

assert int(ans.str.contains("[A-Z]").sum()) == 0, "大文字が残っています"
assert ans.iloc[0] == "taro.yamada@example.com", ans.iloc[0]
print("OK")

<details>
<summary>答え</summary>

```python
ans = raw["メール"].str.lower()
```

`.str` を挟むと、文字列メソッドを列の全要素にまとめて当てられます。
実際に使うときは `raw["メール"] = raw["メール"].str.lower()` と入れ直します
(この教材の最後にまとめる関数では、そうしています)。

</details>

---

## 3. 型を決める

文字列のままでは、**足せませんし、比べられません**。`"320000" + "298000"` は
`"320000298000"` になりますし、`"2019/04/01" < "2019年4月1日"` に意味はありません。

3つ直します。

| 列 | 何が起きているか | 直し方 |
| --- | --- | --- |
| `社員コード` | Excel を通って先頭ゼロが落ちた (`0002` → `2`) | 桁数を決めて `zfill(4)` で埋め直す |
| `入社日` | `2019/04/01` と `2019年4月1日` が混在 | 書式ごとに読んで、成功したほうを採る |
| `基本給` | `￥` とカンマが付いている | 記号を落としてから数値にする |

`errors="coerce"` は「読めなかったら例外ではなく欠損にする」という指定です。
**1行の異常で全体が止まらないようにするため**に使います。
`Int64`(大文字) は欠損を持てる整数型で、`int64`(小文字) は持てません。

> 出力の見どころ: `社員コード` が4桁で揃ったこと。`入社日` の dtype が `datetime64[ns]`、
> `基本給` が `Int64` になったこと。

In [ ]:
# 桁数を決めて、先頭ゼロを埋め直す
raw["社員コード"] = raw["社員コード"].str.zfill(4)

# 書式ごとに読む。読めなかった行は NaT になるので、もう片方で埋める
d1 = pd.to_datetime(raw["入社日"], format="%Y/%m/%d", errors="coerce")
d2 = pd.to_datetime(raw["入社日"], format="%Y年%m月%d日", errors="coerce")
raw["入社日"] = d1.fillna(d2)

# 記号とカンマを落としてから数値にする
s = raw["基本給"].str.replace(r"[¥,]", "", regex=True)
raw["基本給"] = pd.to_numeric(s, errors="coerce").astype("Int64")

print(raw[["社員コード", "氏名", "入社日", "基本給"]].head(5))

print("\n型:")
print(raw.dtypes)

In [ ]:
# ✍ 書いてみる: 週勤務時間 を Int64 にしたものを ans に入れてください。
#              (基本給と同じことを、記号を落とす手順だけ抜いてやります)

ans = ...   # ここに書く

assert str(ans.dtype) == "Int64", f"Int64 のはずです: {ans.dtype}"
assert int(ans.isna().sum()) == 1, f"欠損は1件のはずです: {int(ans.isna().sum())}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = pd.to_numeric(raw["週勤務時間"], errors="coerce").astype("Int64")
```

`４０` が `40` になっているのは、2節の NFKC が効いているからです。
先に表記を揃えておくと、ここが1行で済みます。

</details>

---

## 4. 重複を落とす

重複には**性質の違う2種類**があります。混ぜて消すと、消してはいけない行が消えます。

| | 何が起きたか | どうするか |
| --- | --- | --- |
| 全部の列が同じ | エクスポートの事故。同じ行が2回出た | どちらを残しても同じ。片方を落とす |
| 社員コードだけ同じ | **訂正**。あとから直した行が届いた | **新しいほう**を残す |

新しいほうを選ぶには「どちらが新しいか」を書いた列が要ります。この名簿では `更新日` です。
`更新日` は `2024-04-01` の形 (ISO 8601) なので、**文字列のまま並べ替えても正しく並びます**。

> 出力の見どころ: 落ちるのが 0004 の重複行と、0017 の古いほう (基本給 345000) の2行であること。

In [ ]:
print("全部の列が同じ行:")
print(raw[raw.duplicated(keep=False)][["社員コード", "氏名", "基本給", "更新日"]])

dedup = raw.drop_duplicates()
print("\n完全重複を落とした:", len(raw), "->", len(dedup))

print("\n社員コードだけが同じ行:")
print(dedup[dedup.duplicated(subset="社員コード", keep=False)][["社員コード", "氏名", "基本給", "更新日"]])

# 更新日で並べて、あとのほう (keep="last") を残す
dedup = (
    dedup.sort_values("更新日")
    .drop_duplicates(subset="社員コード", keep="last")
    .sort_values("社員コード")
    .reset_index(drop=True)
)
print("\n訂正行を寄せた:", len(dedup), "行")

---

## 5. 部署を名寄せして結合する

`営業部` `営業` `セールス` `Sales` は、全部 D01 のことです。
NFKC では揃いません。**どれとどれが同じかは、業務上の知識でしか決められない**からです。
そこで、寄せ先を書いた辞書 (`ALIAS`) を人間が用意します。

寄せてから部署マスタを結合します。ここで**結合の三点セット**を必ず書きます。

| 引数 | 何のため |
| --- | --- |
| `how="left"` | 左 (名簿) の行は全部残す。マスタに無くても消さない |
| `validate="m:1"` | **右が1行であることを保証する**。マスタが重複していたら例外で止まる |
| `indicator=True` | どの行が右と一致したかを `_merge` 列で教えてもらう |

**結合は、黙って行が増える事故が起きやすい操作です。** `validate` はその見張りで、
最後の `assert` は「実際に増えなかった」という確認です。

> 出力の見どころ: 結合の前後で行数が 18 のまま変わらないこと。
> `_merge` に `left_only` が混ざっていること。

In [ ]:
# 表記の揺れを、正しい部署コードに寄せる辞書。ここは人間が決める
ALIAS = {
    "営業部": "D01", "営業": "D01", "セールス": "D01", "Sales": "D01",
    "開発部": "D02", "開発": "D02", "エンジニアリング": "D02",
    "人事部": "D03", "人事": "D03",
    "経理部": "D04", "経理": "D04",
    "情報システム部": "D05", "情シス": "D05",
}
dedup["dept_cd"] = dedup["部署"].replace(ALIAS)
print(dedup[["部署", "dept_cd"]].drop_duplicates().sort_values("dept_cd"))

dept = pd.read_csv("/data/dept.csv", dtype=str, keep_default_na=False)

before = len(dedup)
fact = dedup.merge(
    dept[["dept_cd", "dept_name", "division"]],
    on="dept_cd", how="left", validate="m:1", indicator=True,
)
assert len(fact) == before, f"結合で行が増えた: {before} -> {len(fact)}"

print("\n結合の前後:", before, "->", len(fact))
print(fact["_merge"].value_counts())

In [ ]:
# ✍ 書いてみる: 部署マスタに無かった行が何行あるか数えてください
#   (ヒント: そういう行は fact["_merge"] が "left_only" になっています)

ans = ...   # ここに書く

assert ans == 1, f"マスタに無い行の数が違います: {ans!r}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = int((fact["_merge"] == "left_only").sum())
```

`広報部` の1行です。マスタに載っていないだけで、社員が居ないわけではありません。
**行ごと落とさずに残します**(次の節で `未分類` にします)。

</details>

---

## 6. 使えない行を隔離して、集計する

**使えない行は、捨てずに分けます。** 捨ててしまうと、
「合計が合わない」と言われたときに何が抜けたのかを説明できません。

何をもって使えないとするかは、先に文章で決めてから式にします。この名簿では2つです。

- **社員コードが無い** … 誰の行か分からないので、他の表と突き合わせられない
- **入社日が読めない** … 在籍期間が出せない

最後に検算します。`data/` は全部で20行しかないので、
**CSV を開いて目で数えれば、集計結果が正しいか確かめられます**。
この大きさにしてあるのはそのためです。

> 出力の見どころ: `clean 16行 / rejected 2行`。
> 隔離されたのが 0015 (入社日が `不明`) と、社員コードが空の行であること。

In [ ]:
# マスタに無かった部署は、行ごと落とさずに「未分類」として残す
fact["division"] = fact["division"].fillna("未分類")
fact = fact.drop(columns="_merge")

bad = fact["社員コード"].isna() | fact["入社日"].isna()
clean = fact[~bad].reset_index(drop=True)
rejected = fact[bad].reset_index(drop=True)

assert len(clean) + len(rejected) == len(fact), "隔離で行が消えた"
print(f"clean {len(clean)}行 / rejected {len(rejected)}行")
print(rejected[["社員コード", "氏名", "部署", "入社日", "基本給"]])

summary = (
    clean.groupby("division")
    .agg(人数=("社員コード", "size"), 基本給合計=("基本給", "sum"))
    .reset_index()
)
print()
print(summary)

# 検算。CSV を手で数えた値と突き合わせる
assert int(summary["人数"].sum()) == 16, summary
assert int(summary["基本給合計"].sum()) == 4978000, summary
print("\nOK")

---

## この教材で分かったこと

| | |
| --- | --- |
| `dtype=str` / `keep_default_na=False` | 読むときは型も欠損も推測させない。**何が欠損かは自分で決める** |
| NFKC | 全角の英数字・カタカナ・スペースを半角に寄せる。集計の前に必ず通す |
| `errors="coerce"` | 読めない値を例外ではなく欠損にする。1行の異常で全体を止めない |
| `Int64` と `int64` | 大文字のほうは欠損を持てる。数量や金額はこちら |
| `zfill` | Excel を通って落ちた先頭ゼロを埋め直す |
| 重複の2種類 | 全列が同じ = 事故 / キーだけ同じ = 訂正。**後者は新しいほうを残す** |
| 名寄せ辞書 | どれとどれが同じかは業務知識。正規化では揃わない |
| 結合の三点セット | `how` / `validate` / `indicator`。**行が増える事故を止める** |
| `rejected` | 使えない行は捨てずに分ける。捨てると差分を説明できなくなる |
| 検算 | 手で数えられる大きさのデータで、集計結果を必ず突き合わせる |

ここまでを1つの関数にまとめると、次のセルのようになります。
**この形が、そのまま日々動かすスクリプトの中身になります。**

In [ ]:
NA_TOKENS = ["", "-", "N/A", "不明"]

ALIAS = {
    "営業部": "D01", "営業": "D01", "セールス": "D01", "Sales": "D01",
    "開発部": "D02", "開発": "D02", "エンジニアリング": "D02",
    "人事部": "D03", "人事": "D03",
    "経理部": "D04", "経理": "D04",
    "情報システム部": "D05", "情シス": "D05",
}


def norm(s):
    return unicodedata.normalize("NFKC", s).strip()


def clean_roster(path="/data/roster_2024-04.csv", dept_path="/data/dept.csv"):
    """名簿CSVを1枚読んで、使える行 (clean) と隔離した行 (rejected) に分ける"""
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    for col in df.columns:
        df[col] = df[col].map(norm)
    df = df.replace(NA_TOKENS, pd.NA)

    df["メール"] = df["メール"].str.lower()
    df["社員コード"] = df["社員コード"].str.zfill(4)
    d1 = pd.to_datetime(df["入社日"], format="%Y/%m/%d", errors="coerce")
    d2 = pd.to_datetime(df["入社日"], format="%Y年%m月%d日", errors="coerce")
    df["入社日"] = d1.fillna(d2)
    for col in ("基本給", "週勤務時間"):
        s = df[col].str.replace(r"[¥,]", "", regex=True)
        df[col] = pd.to_numeric(s, errors="coerce").astype("Int64")

    df = (
        df.drop_duplicates()
        .sort_values("更新日")
        .drop_duplicates(subset="社員コード", keep="last")
        .sort_values("社員コード")
        .reset_index(drop=True)
    )

    df["dept_cd"] = df["部署"].replace(ALIAS)
    dept = pd.read_csv(dept_path, dtype=str, keep_default_na=False)
    before = len(df)
    df = df.merge(
        dept[["dept_cd", "dept_name", "division"]],
        on="dept_cd", how="left", validate="m:1",
    )
    assert len(df) == before, f"結合で行が増えた: {before} -> {len(df)}"
    df["division"] = df["division"].fillna("未分類")

    bad = df["社員コード"].isna() | df["入社日"].isna()
    return df[~bad].reset_index(drop=True), df[bad].reset_index(drop=True)


clean, rejected = clean_roster()
assert (len(clean), len(rejected)) == (16, 2), (len(clean), len(rejected))
print(f"clean {len(clean)}行 / rejected {len(rejected)}行")
print("OK")

---

## 次にやること

この教材で触ったのは、パイプラインのうち**真ん中のクレンジングだけ**です。
前後にはまだ続きがあります。

| | |
| --- | --- |
| **tutorial-01 POS売上パイプライン** | 同じことを、複数ファイルの取り込みから Parquet への出力まで通しでやります。文字コードの違い、税区分、**二度流しても壊れない書き込み**まで扱います |
| **quest-01 raw-ingest** | 仕様書だけ渡されて、自力で実装する腕試しです |
| **drills/pandas-00〜03** | 個々の操作でつまずいたときに戻る場所です |

読み物のほうは `docs/03 データは汚い` が、この教材の裏付けになっています。